# Курс Data Science. Дипломный проект. Задача классификации.
## «Модель прогнозирования остановки на питстоп в чемпионате автогонок в классе "Формула1"»

## 4. Реализация полученной предсказательной модели в продакшене.

In [7]:
import pandas as pd
import pickle

#отключим оповещения
import warnings
warnings.filterwarnings("ignore")

Загрузим нашу предсказательную модель, кодировщик и стандартизатор а также набор тестовых данных.

In [8]:
# Загрузка сохраненной модели из файла pickle
with open("web/app/models/best_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)
# Загрузка стандартизатора из файла pickle
with open("web/app/models/mm_scaler.csv", "rb") as f:
    mm_scaler = pickle.load(f)
# Загрузка кодировщика из файла pickle    
with open("web/app/models/encoder.csv", "rb") as f:
    encoder = pickle.load(f)

In [9]:
df = pd.read_csv('data/f1_strategy_dataset_v6.csv')

# Выбираем 30 строк из датасета
df_test = df.iloc[:30]
df_test.head(2)


,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change,Prev_TyreLife
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0.0,0.017241,0.117647,0.0,NaN
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0.0,0.034483,0.176471,-1.0,2.0


Напишем функцию, которая произведет все те же преобразования с тестовыми данными, что ми провели с основным датсетом для обучения модели.

In [10]:
def transform_data(df_input, mm_scaler, encoder):
    df_out = df_input.copy()
    df_out.fillna(0, inplace=True) # на всякий случай меняем NaN на 0
    df_out = df_out.drop(['Prev_TyreLife'], axis=1)
    df_out = df_out.drop(['RaceProgress'], axis=1)
    df_out['Year'] = df_out['Year'].astype(object)
    race_length = df_out.groupby('Race')['LapNumber'].transform("max")
    df_out['RemainingLaps'] = race_length - df_out['LapNumber']
    df_out['TyreLifeRemainingRatio'] = df_out['TyreLife'] / (df_out['RemainingLaps']+1)
    df_out['PositionTyreLife'] = df_out['Position'] * df_out['TyreLife']
    df_out['LapDelta_TyreLife'] =  df_out['LapTime_Delta'] * df_out['TyreLife']
    df_out['Stint_TyreLife'] = df_out['Stint'] * df_out['TyreLife']

    cat_cols = df_out.select_dtypes(include=['object']).columns.tolist()

    type_bin = encoder.transform(df_out[cat_cols]) # создаем закодированные признаки
    df_out = pd.concat([df_out, type_bin], axis=1) # добавляем новые признаки в датафрейм
    df_out.drop(cat_cols, axis=1, inplace=True) # удаляем старые ненужные

    col_names = list(df_out.columns)
    df_out = mm_scaler.transform(df_out)
    df_out = pd.DataFrame(df_out, columns=col_names)
    return df_out

Запустим и проверим результат работы этой функции.

In [11]:
df_out_transf = transform_data(df_test, mm_scaler, encoder)
df_out_transf.head()

,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,Normalized_TyreLife,...,Race_United States Grand Prix,Race_Chinese Grand Prix,Race_Emilia Romagna Grand Prix,Race_French Grand Prix,Race_Pre-Season Test,Race_Pre-Season Track Session,Year_2023.0,Year_2024.0,Year_2025.0,Year_2022.0
0,0.000000,0.0,0.012987,0.842105,0.156287,0.513789,0.582499,0.0,0.0,0.106188,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.012987,0.0,0.025974,0.894737,0.123438,0.496101,0.567511,0.0,0.0,0.165775,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.025974,0.0,0.038961,0.894737,0.115106,0.509302,0.563709,0.0,0.0,0.225363,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.038961,0.0,0.051948,0.894737,0.114283,0.513346,0.563333,0.0,0.0,0.284950,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.051948,0.0,0.064935,0.894737,0.113497,0.513366,0.562975,0.0,0.0,0.344538,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


Сделаем предсказание нашей загруженной модели на тестовых данных. Для визуализации результата работы модели выведем предсказания и правильные значения классов целевой переменной.

In [12]:
X_test = df_out_transf.drop(['PitNextLap'], axis=1)
y_test_pred_loaded = loaded_model.predict(X_test)
y_test_pred_loaded_proba = loaded_model.predict_proba(X_test)
print('PitNextLap в исходном датасете  : {}'.format(df_test['PitNextLap'].values.astype(int)))
print('Предсказанный признак PitNextLap: {}'.format(y_test_pred_loaded))

PitNextLap в исходном датасете  : [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Предсказанный признак PitNextLap: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


Убедились в корректной работе предсказательной модели.

### Контейнеризация Docker

Образ, который мы будем использовать для своего контейнера — tiangolo/uwsgi-nginx-flask.  
Образ этот создан на базе ОС Linux, и в нём уже настроено взаимодействие серверов uWSGI и NGINX через сокеты.

Вся необходимая структура и сам конфигурационный файл для создания Docker образа находится в директории web.

Для создания образа в терминале ПК набираем команду:

    docker build -t diplom .

Ждем окончания установки всех необходимых библиотек и запускаем нашь сервер командой

    docker run -p 5000:5000 diplom python server.py

после запуска страница браузера по адресу 127.0.0.1:5000 выдаст сообщение "Тестовое сообщение. Сервер запущен!"

далее нужно перейти в терминале в директорию test 

    cd test
    
и запустить скрипт отправки запроса client.py

    python3 client.py

Меняя в строке запроса в файле client.py параметры входных данных, мы будем получать новое предсказание.

В итоге имеем рабочий сервис с обученной моделью по предсказания остановки на питстоп.

### Краткие выводы по выводу модели в продакшен.

В это части дипломной работы мы проделали следующее:
* Загрузили предсказательную модель XGBClassifier и данные для ее тестирования.
* Провели проверка модель на воспроизводимость.
* Создали Docker контейнер.
* Составили инструкцию по созданию и запуску контейнера.